# LEN GNN Rakip Deneyi
Bu notebook açıklanabilir toplu graf özellikleri tabanı ile GCN modelini bağımsız grafik ayrımında karşılaştırır. Çıktı ancak gerçek LEN-Small arşivi, SHA-256 kaydı ve tamamlanmış tekrarlarla rapor sonucu sayılır.

In [ ]:
%pip install -q 'torch-geometric>=2.7,<3.0' 'scikit-learn>=1.7,<2.0' 'networkx>=3.5,<4.0'


In [ ]:
import subprocess
import sys
import tarfile
from pathlib import Path

WORKSPACE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPOSITORY = WORKSPACE / 'Sagduyu'
if not REPOSITORY.exists():
    subprocess.run(
        [
            'git',
            'clone',
            'https://github.com/FARADTechnologies/Sagduyu.git',
            str(REPOSITORY),
        ],
        check=True,
    )
DATASET_URL = 'https://cse.buffalo.edu/~erdem/small_encoder_final.tar'
ARCHIVE = WORKSPACE / 'len_small.tar'
EXTRACT_ROOT = WORKSPACE / 'len_small'
OUTPUT = WORKSPACE / 'len_results.json'
subprocess.run([
    'curl', '--fail', '--location', '--retry', '3', '--continue-at', '-',
    '--output', str(ARCHIVE), DATASET_URL,
], check=True)
def discover_graphs(root):
    return [
        path for path in root.rglob('*.json')
        if path.stem.lower().endswith(('_campaign_fulldata', '_noncampaign_fulldata'))
    ]
graph_files = discover_graphs(EXTRACT_ROOT)
if not graph_files:
    EXTRACT_ROOT.mkdir(exist_ok=True)
    with tarfile.open(ARCHIVE) as archive:
        archive.extractall(EXTRACT_ROOT, filter='data')
    graph_files = discover_graphs(EXTRACT_ROOT)
graph_directories = {path.parent for path in graph_files}
assert len(graph_directories) == 1, 'Graf dosyaları tek bir dizinde bulunmalıdır.'
assert len(graph_files) >= 4, 'En az dört etiketli graf bulunmalıdır.'
DATASET_DIR = graph_directories.pop()


In [ ]:
subprocess.run([
    sys.executable, str(REPOSITORY / 'experiments/len_gnn_challenger.py'),
    '--dataset-dir', str(DATASET_DIR),
    '--archive', str(ARCHIVE),
    '--output', str(OUTPUT),
    '--seeds', '11,23,37',
    '--epochs', '40',
], cwd=REPOSITORY, check=True)


In [ ]:
import json

result = json.loads(OUTPUT.read_text())
print(json.dumps(result['mean'], indent=2))
print('GNN kabul kapısı:', result['acceptance_gate']['accepted'])
